# 06. Embedding Visualization — t-SNE / UMAP
# 06. Embedding 시각화 — t-SNE / UMAP

**Role / 담당**: R3 (Evaluation & Reporting)  
**Stage**: S1 Prototype (W3~W4)  
**Purpose / 목적**: Visualize embeddings extracted from Phase 0 (Contrastive Learning) or Phase 2 backbone in 2D to visually verify the validity of Grayspot Level boundaries.  
Phase 0 (Contrastive Learning) 또는 Phase 2 backbone에서 추출한 embedding을 2D로 투영하여 Grayspot Level 경계의 타당성을 시각적으로 검증한다.

---

## Role of This Notebook / 이 노트북의 역할 (PRD 참조)

| Reference / 참조 섹션 | Content / 내용 |
|-----------|------|
| PRD §3.2 | Phase 1 label refinement: Level boundary review based on embedding visualization / Phase 1 기준 정제: embedding 시각화 기반 Level 경계 검토 |
| PRD §5.6.4 | Representation Analysis: t-SNE/UMAP, per-color distribution comparison, label-embedding mismatch analysis / Representation Analysis: t-SNE/UMAP, 색상별 분포 비교, 라벨-Embedding 불일치 분석 |
| PRD §3.2.3 | Refinement completion criteria: ARI ≥ 0.6, mismatch samples ≤ 10% / 정제 완료 판단: ARI ≥ 0.6, 불일치 샘플 ≤ 10% |
| PRD §5.3.1 | Per-color independent model separation criteria: Silhouette Score ≥ 0.3 / 색상별 독립 모델 분리 판단: Silhouette Score ≥ 0.3 |

## Output Artifacts / 출력 산출물
- `data_set/embedding_viz/tsne_by_level.html` — Level-colored t-SNE (interactive) / Level별 색상 코딩 t-SNE
- `data_set/embedding_viz/umap_by_level.html` — Level-colored UMAP (interactive) / Level별 색상 코딩 UMAP
- `data_set/embedding_viz/tsne_by_color.html` — CMYK per-channel distribution t-SNE / CMYK 색상별 분포 t-SNE
- `data_set/embedding_viz/metrics_summary.json` — Quantitative metrics: ARI, Silhouette Score / ARI, Silhouette Score 등 정량 지표
- `data_set/embedding_viz/mismatch_samples.csv` — Label-embedding mismatch sample list / 라벨-Embedding 불일치 샘플 목록

> **S1 Prototype principle / S1 프로토타입 원칙**: Working visualization takes priority over perfect modularization.  
> 완벽한 모듈화보다 "돌아가는" 시각화가 우선이다.  
> If no model checkpoint exists, the pipeline can be verified using random dummy embeddings.  
> 모델 checkpoint가 없는 경우 랜덤 더미 embedding으로 파이프라인을 검증할 수 있다.

---
## 0. Environment Setup / 환경 설정 및 의존성

In [ ]:
# 필요 패키지 설치 (최초 1회) / Install required packages (first time only)
# !pip install umap-learn scikit-learn plotly pandas numpy torch torchvision

In [3]:
# Python 3.11.5 기준 / Python 3.11.5 standard
import os
import json
import warnings
from typing import Optional, Dict, List, Tuple
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import LabelEncoder

try:
    import umap
    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False
    print("[경고 / Warning] umap-learn 미설치 / not installed → UMAP 시각화 건너뜀 / Skipping UMAP. `pip install umap-learn`")

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from PIL import Image
from pathlib import Path

print(f"Python 3.11.5 기준 노트북 / Python 3.11.5 standard notebook")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA 사용 가능 / Available: {torch.cuda.is_available()}")
print(f"UMAP 사용 가능 / Available: {UMAP_AVAILABLE}")

Python 3.11.5 기준 노트북
PyTorch: 2.11.0
CUDA 사용 가능: False
UMAP 사용 가능: True


---
## 1. Settings / 설정

> Paths and values are set directly without config.yaml. / config.yaml 없이 직접 경로와 값을 지정한다.

In [ ]:
from pathlib import Path
import os

# ── 경로 설정 / Path settings ─────────────────────────────────────────
ROOT_DIR    = Path('../..').resolve()                    # CMYK_MAIN/
LABELED_DIR = ROOT_DIR / 'data_set' / 'labeled'         # 라벨링 폴더 / Labeled folder
PATCH_DIR   = ROOT_DIR / 'data_set' / 'roi'             # ROI 패치 디렉토리 / ROI patch directory
LABELS_CSV  = ROOT_DIR / 'data_set' / 'labels_v0.csv'  # 라벨 CSV / Label CSV
OUTPUT_DIR  = ROOT_DIR / 'data_set' / 'embedding_viz'  # 출력 폴더 / Output folder

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── 데이터 설정 / Data settings ───────────────────────────────────────
CHANNELS   = ['Y', 'M', 'C', 'K']
NUM_LEVELS = 6        # Level 0 ~ 5
IMAGE_SIZE = 224      # 모델 입력 크기 / Model input size

# CSV 컬럼명 매핑 / CSV column name mapping
COLOR_COLUMNS = {
    'Y': 'Y',
    'M': 'M',
    'C': 'C',
    'K': 'K',
}

# 고정 ROI 좌표 (x, y, w, h) / Fixed ROI coordinates
ROI_COORDS = {
    'Y': [0,   0,   800, 200],
    'M': [0,   200, 800, 200],
    'C': [0,   400, 800, 200],
    'K': [0,   600, 800, 200],
}

# ── 모델 설정 / Model settings ────────────────────────────────────────
BACKBONE_NAME  = 'efficientnet_b0'
EMBEDDING_DIM  = 1280   # EfficientNet-B0 출력 차원 / Output dimension
BATCH_SIZE     = 32

# ── t-SNE / UMAP 설정 ─────────────────────────────────────────────────
TSNE_PARAMS = {'n_components': 2, 'perplexity': 30, 'n_iter': 1000, 'random_state': 42}
UMAP_PARAMS = {'n_components': 2, 'n_neighbors': 15, 'min_dist': 0.1, 'random_state': 42}

# ── 분석 임계값 / Analysis thresholds ────────────────────────────────
ARI_THRESHOLD        = 0.6
MISMATCH_THRESHOLD   = 0.10
SILHOUETTE_THRESHOLD = 0.3

# ── 시각화 색상 / Visualization colors ───────────────────────────────
LEVEL_COLORS = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c', '#9b59b6', '#1a1a2e']
CMYK_COLORS  = {'Y': '#f5e642', 'M': '#e91e8c', 'C': '#00b4d8', 'K': '#444444'}

# ── 확인 출력 / Verification output ──────────────────────────────────
print(f'ROOT_DIR   : {ROOT_DIR}')
print(f'LABELED_DIR: {LABELED_DIR}')
print(f'LABELS_CSV : {LABELS_CSV}')
print(f'CSV exists / 파일 존재: {LABELS_CSV.exists()}')

print('\nLabeled folder counts / 라벨링 폴더 샘플 수:')
for ch in CHANNELS:
    for lv in range(NUM_LEVELS):
        folder = LABELED_DIR / ch / str(lv)
        count  = len(list(folder.glob('*'))) if folder.exists() else 0
        print(f'  [{ch}] Level {lv}: {count}개')

ROOT_DIR   : /Users/yangjin-hyeong/Desktop/CMYK_MAIN
LABELED_DIR: /Users/yangjin-hyeong/Desktop/CMYK_MAIN/data_set/labeled
LABELS_CSV : /Users/yangjin-hyeong/Desktop/CMYK_MAIN/data_set/labels_v0.csv
CSV exists / 파일 존재: True

Labeled folder counts / 라벨링 폴더 샘플 수:
  [Y] Level 0: 7개
  [Y] Level 1: 23개
  [Y] Level 2: 39개
  [Y] Level 3: 38개
  [Y] Level 4: 5개
  [Y] Level 5: 0개
  [M] Level 0: 14개
  [M] Level 1: 137개
  [M] Level 2: 228개
  [M] Level 3: 472개
  [M] Level 4: 200개
  [M] Level 5: 73개
  [C] Level 0: 0개
  [C] Level 1: 16개
  [C] Level 2: 34개
  [C] Level 3: 89개
  [C] Level 4: 32개
  [C] Level 5: 2개
  [K] Level 0: 5개
  [K] Level 1: 16개
  [K] Level 2: 71개
  [K] Level 3: 231개
  [K] Level 4: 121개
  [K] Level 5: 2개


---
## 2. Data Load / 데이터 로드

### 2.1 Label CSV Load / 라벨 CSV 로드

In [14]:
def load_labels(labels_csv, color_columns: dict) -> pd.DataFrame:
    """
    라벨 CSV를 로드하여 long-format DataFrame으로 변환한다.
    Loads the label CSV and converts it to a long-format DataFrame.

    원본 CSV 형식 (wide) / Original CSV format (wide):
        filename, C, M, Y, K

    반환 형식 (long) / Return format (long):
        filename, color, level, confidence
        - 각 이미지당 4개 행 (Y/M/C/K) / 4 rows per image
    """
    df = pd.read_csv(labels_csv)
    print(f"[CSV 로드 / Loaded] 원본 행 수 / Rows: {len(df)}, 컬럼 / Columns: {list(df.columns)}")

    # Wide → Long 변환 / Wide to Long conversion
    records = []
    for _, row in df.iterrows():
        for color_code, col_name in color_columns.items():
            if col_name not in df.columns:
                continue  # 컬럼 없으면 건너뜀 / Skip if column not found
            record = {
                "filename":   row["filename"],
                "color":      color_code,
                "level":      int(row[col_name]),
                "confidence": row.get("confidence", "확실"),
            }
            records.append(record)

    long_df = pd.DataFrame(records)
    print(f"[변환 완료 / Converted] Long-format 행 수 / Rows: {len(long_df)}")
    if len(long_df) > 0:
        print(long_df.groupby(["color", "level"]).size().unstack(fill_value=0))
    return long_df


def create_dummy_labels(n_per_class: int = 20) -> pd.DataFrame:
    """
    S1 프로토타입: 실제 라벨 CSV가 없을 때 사용하는 더미 데이터.
    S1 Prototype: Dummy data used when real label CSV is not available.
    Level 0~5, 색상 Y/M/C/K 조합으로 균일하게 생성.
    Generated uniformly for Level 0~5 and color Y/M/C/K combinations.
    """
    print("[더미 데이터 / Dummy] 실제 CSV 미존재 → 더미 데이터 생성 / CSV not found → generating dummy data")
    records = []
    colors  = ["Y", "M", "C", "K"]
    levels  = [0, 1, 2, 3, 4, 5]
    idx     = 0
    for c in colors:
        for lv in levels:
            for i in range(n_per_class):
                records.append({
                    "filename":   f"scan_dummy_{idx:04d}.png",
                    "color":      c,
                    "level":      lv,
                    "confidence": "확실" if np.random.rand() > 0.2 else "불확실",
                })
                idx += 1
    return pd.DataFrame(records)


# 실제 CSV 존재 여부에 따라 분기 / Branch based on CSV existence
if LABELS_CSV.exists():
    df_labels = load_labels(LABELS_CSV, COLOR_COLUMNS)
else:
    df_labels = create_dummy_labels(n_per_class=20)

print(f"\n총 샘플 수 / Total samples: {len(df_labels)}")
df_labels.head()

[CSV 로드] 원본 행 수: 1855, 컬럼: ['filename', 'C', 'M', 'Y', 'K']
[변환 완료] Long-format 행 수: 0


KeyError: 'color'

### 2.2 Embedding Extraction (Backbone → GAP) / Embedding 추출 (Backbone → GAP)

> **PRD §3.0**: If Phase 0 checkpoint exists, load those backbone weights; otherwise use ImageNet pretrained weights.  
> Phase 0 checkpoint가 있으면 해당 backbone을 로드하고, 없으면 ImageNet pretrained 사용.

In [ ]:
# ── 전처리 변환 / Preprocessing transform ────────────────────────────────
# S1 프로토타입에서는 단순 ImageNet 정규화 사용 / S1 prototype uses simple ImageNet normalization
# S2 모듈화 이후: preprocessing.py 의 SSOT 전처리로 교체 / Replace with SSOT preprocessing.py after S2
inference_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.Grayscale(num_output_channels=3),   # 단일채널 → 3채널 복제 (PRD §6.8.4) / Single channel → 3ch replication
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),  # ImageNet 기준 (S1 임시) / ImageNet standard (S1 temporary)
])


class PatchDataset(Dataset):
    """
    색상 패치 이미지 Dataset.
    Color patch image Dataset.

    폴더 구조 / Folder structure:
        data_set/labeled/{color}/{level}/*.png
    """

    def __init__(self, df: pd.DataFrame, patch_dir: Path, transform):
        self.df        = df.reset_index(drop=True)
        self.patch_dir = Path(patch_dir)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        color    = row["color"]
        fname    = row["filename"]
        level    = row["level"]

        # data_set/labeled/{color}/{level}/{filename} 경로 구성
        # Build path: data_set/labeled/{color}/{level}/{filename}
        img_path = self.patch_dir / color / str(level) / fname

        if img_path.exists():
            img = Image.open(img_path).convert("RGB")
        else:
            # 이미지 미존재 시 더미 이미지 (파이프라인 검증용)
            # Dummy image when file not found (for pipeline verification)
            img = Image.fromarray(
                np.random.randint(100, 200, (IMAGE_SIZE, IMAGE_SIZE, 3), dtype=np.uint8)
            )

        return self.transform(img), idx  # tensor, 원본 인덱스 반환 / Returns tensor and original index


def build_backbone(backbone_name: str, checkpoint, device: torch.device) -> nn.Module:
    """
    Backbone 모델을 로드하고 분류기 head를 제거하여 embedding 추출기로 반환한다.
    Loads backbone model, removes classifier head, and returns it as a feature extractor.

    PRD §3.0:
        Phase 0 checkpoint 있으면 해당 weights 로드 / Load Phase 0 checkpoint weights if available
        없으면 ImageNet pretrained 사용 / Otherwise use ImageNet pretrained
    """
    if backbone_name == "efficientnet_b0":
        model = models.efficientnet_b0(weights="IMAGENET1K_V1")
        model.classifier = nn.Identity()   # classifier head 제거 / Remove classifier head
    elif backbone_name == "resnet18":
        model = models.resnet18(weights="IMAGENET1K_V1")
        model.fc = nn.Identity()
    elif backbone_name == "resnet34":
        model = models.resnet34(weights="IMAGENET1K_V1")
        model.fc = nn.Identity()
    else:
        raise ValueError(f"지원하지 않는 backbone / Unsupported backbone: {backbone_name}")

    if checkpoint is not None and os.path.exists(str(checkpoint)):
        state = torch.load(checkpoint, map_location=device)
        # Phase 0 checkpoint는 projection_head 포함 가능 → backbone 키만 선택적 로드
        # Phase 0 checkpoint may include projection_head → selectively load backbone keys only
        backbone_state = {k.replace("backbone.", ""): v
                          for k, v in state.items() if k.startswith("backbone.")}
        if backbone_state:
            model.load_state_dict(backbone_state, strict=False)
            print(f"[체크포인트 / Checkpoint] backbone weights 로드 / Loaded: {checkpoint}")
        else:
            model.load_state_dict(state, strict=False)
            print(f"[체크포인트 / Checkpoint] 전체 weights 로드 / Full weights loaded: {checkpoint}")
    else:
        print("[체크포인트 없음 / No checkpoint] ImageNet pretrained weights 사용 / Using ImageNet pretrained weights")

    model.eval()  # 추론 모드 — BN, Dropout 비활성화 / Inference mode — deactivate BN, Dropout
    return model.to(device)


@torch.no_grad()
def extract_embeddings(model: Optional[nn.Module],
                       dataloader: Optional[DataLoader],
                       device: torch.device,
                       use_dummy: bool = False,
                       n_samples: int = 0,
                       embed_dim: int = 1280) -> np.ndarray:
    """
    모델에서 embedding vector를 일괄 추출한다.
    Extracts embedding vectors from the model in batch.

    use_dummy=True 이면 랜덤 embedding 생성 (이미지 미존재 시 파이프라인 검증용)
    If use_dummy=True, generates random embeddings (for pipeline verification when images are missing)
    """
    if use_dummy:
        print(f"[더미 Embedding / Dummy] {n_samples}개 랜덤 벡터 생성 / Generating random vectors (dim={embed_dim})")
        return np.random.randn(n_samples, embed_dim).astype(np.float32)

    embeddings = []
    for batch_tensors, _ in dataloader:
        batch_tensors = batch_tensors.to(device)
        feats         = model(batch_tensors)   # (B, embed_dim)
        embeddings.append(feats.cpu().numpy())

    return np.concatenate(embeddings, axis=0)


# ── 실행 / Execute ────────────────────────────────────────────────────────
device = torch.device(
    'cuda' if torch.cuda.is_available() else
    'mps'  if torch.backends.mps.is_available() else
    'cpu'
)
print(f"Device: {device}")

# CHECKPOINT 가 None 이면 더미 embedding 사용 / Use dummy embedding if CHECKPOINT is None
CHECKPOINT = None  # Phase 0 backbone 경로 지정 / Set Phase 0 backbone path if available
                   # 예시 / Example: ROOT_DIR / 'data_set' / 'models' / 'phase0_backbone_C.pt'

if LABELED_DIR.exists() and CHECKPOINT is not None:
    # 실제 이미지 + checkpoint 가 모두 있을 때 / When both real images and checkpoint exist
    backbone   = build_backbone(BACKBONE_NAME, CHECKPOINT, device)
    dataset    = PatchDataset(df_labels, LABELED_DIR, inference_transform)
    loader     = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=0, pin_memory=torch.cuda.is_available())
    embeddings = extract_embeddings(backbone, loader, device, use_dummy=False)
else:
    # checkpoint 없으면 더미 embedding 사용 (파이프라인 검증용)
    # Use dummy embeddings when no checkpoint (for pipeline verification)
    print(f"[경고 / Warning] checkpoint 없음 → 더미 embedding 사용 / No checkpoint → using dummy embeddings")
    embeddings = extract_embeddings(
        None, None, device,
        use_dummy=True,
        n_samples=len(df_labels),
        embed_dim=EMBEDDING_DIM
    )

print(f"\nEmbedding shape: {embeddings.shape}  (샘플 수 × embedding 차원 / samples × embedding dim)")

---
## 3. Dimensionality Reduction (t-SNE / UMAP) / 차원 축소

In [ ]:
def run_tsne(embeddings: np.ndarray, params: dict) -> np.ndarray:
    """
    t-SNE 2D 투영. 샘플 수가 많으면 perplexity 자동 조정.
    t-SNE 2D projection. Auto-adjusts perplexity for large sample counts.
    """
    n          = len(embeddings)
    perplexity = min(params["perplexity"], n // 4)  # perplexity < n_samples/4 조건 / Condition
    print(f"[t-SNE] n_samples={n}, perplexity={perplexity}, n_iter={params['n_iter']}")

    tsne = TSNE(
        n_components=params["n_components"],
        perplexity=perplexity,
        n_iter=params["n_iter"],
        random_state=params["random_state"],
        init="pca",
        learning_rate="auto",
        verbose=0,
    )
    proj = tsne.fit_transform(embeddings)
    print(f"[t-SNE 완료 / Done] KL Divergence: {tsne.kl_divergence_:.4f}")
    return proj


def run_umap(embeddings: np.ndarray, params: dict) -> Optional[np.ndarray]:
    """
    UMAP 2D 투영. umap-learn 미설치 시 None 반환.
    UMAP 2D projection. Returns None if umap-learn is not installed.
    """
    if not UMAP_AVAILABLE:
        return None
    print(f"[UMAP] n_neighbors={params['n_neighbors']}, min_dist={params['min_dist']}")
    reducer = umap.UMAP(
        n_components=params["n_components"],
        n_neighbors=params["n_neighbors"],
        min_dist=params["min_dist"],
        random_state=params["random_state"],
        verbose=False,
    )
    proj = reducer.fit_transform(embeddings)
    print("[UMAP 완료 / Done]")
    return proj


print("=" * 50)
print("t-SNE 실행 중 / Running... (수분 소요 가능 / may take several minutes)")
tsne_proj = run_tsne(embeddings, TSNE_PARAMS)

print("=" * 50)
print("UMAP 실행 중 / Running...")
umap_proj = run_umap(embeddings, UMAP_PARAMS)

# 투영 결과를 DataFrame에 추가 / Add projection results to DataFrame
df_viz = df_labels.copy()
df_viz["tsne_x"]    = tsne_proj[:, 0]
df_viz["tsne_y"]    = tsne_proj[:, 1]
df_viz["level_str"] = df_viz["level"].astype(str)  # plotly color 용 / For plotly color mapping

if umap_proj is not None:
    df_viz["umap_x"] = umap_proj[:, 0]
    df_viz["umap_y"] = umap_proj[:, 1]

print("\n차원 축소 완료 / Dimensionality reduction complete.")
df_viz.head()

---
## 4. Visualization / 시각화

### 4.1 t-SNE — Level-colored / Level별 색상 코딩

> **PRD §3.2.2 Step 2**: Overlay existing Level labels to check alignment with clusters.  
> 기존 Level 라벨을 overlay하여 군집과의 일치도 확인

In [ ]:
def plot_projection_by_level(df: pd.DataFrame,
                              x_col: str, y_col: str,
                              method_name: str,
                              level_colors: List[str],
                              output_path: Optional[str] = None) -> go.Figure:
    """
    Level별 색상 코딩 scatter plot.
    Level-colored scatter plot.
    hover에 파일명, 색상, Level, 신뢰도를 표시한다.
    Displays filename, color, Level, and confidence on hover.
    """
    color_map = {str(i): c for i, c in enumerate(level_colors)}

    fig = px.scatter(
        df, x=x_col, y=y_col,
        color="level_str",
        color_discrete_map=color_map,
        symbol="color",                   # CMYK 색상을 마커 모양으로 구분 / Distinguish CMYK by marker shape
        hover_data=["filename", "color", "level", "confidence"],
        category_orders={"level_str": [str(i) for i in range(6)]},
        title=f"{method_name} — Grayspot Level별 Embedding 분포 / Embedding Distribution by Level",
        labels={
            "level_str": "Level (0~5)",
            "color":     "CMYK 채널 / Channel",
        },
        template="plotly_dark",
        width=900, height=650,
    )

    fig.update_traces(marker=dict(size=7, opacity=0.85,
                                   line=dict(width=0.5, color="rgba(255,255,255,0.3)")))
    fig.update_layout(
        legend_title_text="Level",
        font=dict(family="Segoe UI", size=12),
        title_font=dict(size=16),
        margin=dict(l=40, r=40, t=60, b=40),
    )

    if output_path:
        fig.write_html(output_path)
        print(f"[저장 / Saved] {output_path}")

    return fig


fig_tsne_level = plot_projection_by_level(
    df_viz, "tsne_x", "tsne_y",
    method_name="t-SNE",
    level_colors=LEVEL_COLORS,
    output_path=str(OUTPUT_DIR / "tsne_by_level.html"),
)
fig_tsne_level.show()

### 4.2 t-SNE — Per-CMYK Distribution / CMYK 색상별 분포

> **PRD §5.6.4**: Compare embedding distributions by color → determine need for per-color independent models.  
> 색상별 embedding 분포 비교 → 색상별 독립 모델 분리 필요성 판단

In [ ]:
def plot_projection_by_color(df: pd.DataFrame,
                              x_col: str, y_col: str,
                              method_name: str,
                              cmyk_colors: Dict[str, str],
                              output_path: Optional[str] = None) -> go.Figure:
    """
    CMYK 채널별 색상 코딩 scatter plot.
    CMYK channel-colored scatter plot.
    Level을 마커 크기로 표현하여 Level 분포도 함께 확인 가능.
    Level is represented by marker size so Level distribution can also be checked.
    """
    fig = px.scatter(
        df, x=x_col, y=y_col,
        color="color",
        color_discrete_map=cmyk_colors,
        size="level",                     # Level을 마커 크기로 / Level as marker size
        size_max=18,
        hover_data=["filename", "color", "level", "confidence"],
        title=f"{method_name} — CMYK 채널별 Embedding 분포 / Distribution by Channel (마커 크기=Level / Marker size=Level)",
        labels={"color": "CMYK 채널 / Channel"},
        template="plotly_dark",
        width=900, height=650,
    )

    fig.update_traces(marker=dict(opacity=0.75,
                                   line=dict(width=0.5, color="rgba(255,255,255,0.2)")))
    fig.update_layout(
        font=dict(family="Segoe UI", size=12),
        title_font=dict(size=16),
        margin=dict(l=40, r=40, t=60, b=40),
    )

    if output_path:
        fig.write_html(output_path)
        print(f"[저장 / Saved] {output_path}")

    return fig


fig_tsne_color = plot_projection_by_color(
    df_viz, "tsne_x", "tsne_y",
    method_name="t-SNE",
    cmyk_colors=CMYK_COLORS,
    output_path=str(OUTPUT_DIR / "tsne_by_color.html"),
)
fig_tsne_color.show()

### 4.3 UMAP (if installed / 설치된 경우)

In [ ]:
if umap_proj is not None:
    fig_umap_level = plot_projection_by_level(
        df_viz, "umap_x", "umap_y",
        method_name="UMAP",
        level_colors=LEVEL_COLORS,
        output_path=str(OUTPUT_DIR / "umap_by_level.html"),
    )
    fig_umap_level.show()
else:
    print("[건너뜀 / Skipped] UMAP 미설치 / Not installed")

### 4.4 Per-color Subplots — Level Distribution Overview / 색상별 서브플롯 — Level 분포 한눈에 보기

> Compare 4 CMYK channels in a 2×2 subplot layout.  
> 4개 CMYK 채널을 2×2 서브플롯으로 비교

In [ ]:
def plot_per_color_subplot(df: pd.DataFrame,
                            x_col: str, y_col: str,
                            method_name: str,
                            level_colors: List[str]) -> go.Figure:
    """
    CMYK 4채널을 2×2 서브플롯으로 Level 분포 비교.
    Compares Level distribution of 4 CMYK channels in a 2×2 subplot.
    """
    colors_order = ["Y", "M", "C", "K"]
    color_titles = {"Y": "Yellow", "M": "Magenta", "C": "Cyan", "K": "Black"}

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=[color_titles[c] for c in colors_order],
        horizontal_spacing=0.08,
        vertical_spacing=0.12,
    )

    level_color_map  = {i: level_colors[i] for i in range(6)}
    shown_in_legend  = set()

    for idx, color_code in enumerate(colors_order):
        row  = idx // 2 + 1
        col  = idx % 2 + 1
        df_c = df[df["color"] == color_code]

        for level in range(6):
            df_lv       = df_c[df_c["level"] == level]
            if len(df_lv) == 0:
                continue
            show_legend = level not in shown_in_legend
            fig.add_trace(
                go.Scatter(
                    x=df_lv[x_col], y=df_lv[y_col],
                    mode="markers",
                    name=f"Level {level}",
                    legendgroup=f"Level {level}",
                    showlegend=show_legend,
                    marker=dict(
                        color=level_color_map[level],
                        size=6, opacity=0.8,
                        line=dict(width=0.3, color="rgba(255,255,255,0.2)"),
                    ),
                    text=df_lv["filename"],
                    hovertemplate=(
                        f"색상 / Color: {color_code}<br>"
                        "Level: %{marker.color}<br>"
                        "파일 / File: %{text}<extra></extra>"
                    ),
                ),
                row=row, col=col,
            )
            shown_in_legend.add(level)

    fig.update_layout(
        title=f"{method_name} — CMYK 채널별 Level 분포 (4분할) / Level Distribution by Channel",
        title_font=dict(size=16),
        template="plotly_dark",
        font=dict(family="Segoe UI", size=11),
        width=1100, height=800,
        margin=dict(l=40, r=40, t=80, b=40),
    )
    return fig


fig_subplot = plot_per_color_subplot(
    df_viz, "tsne_x", "tsne_y",
    method_name="t-SNE",
    level_colors=LEVEL_COLORS,
)
fig_subplot.write_html(str(OUTPUT_DIR / "tsne_per_color_subplot.html"))
print(f"[저장 / Saved] {OUTPUT_DIR / 'tsne_per_color_subplot.html'}")
fig_subplot.show()

---
## 5. Quantitative Metrics / 정량 지표 계산

### 5.1 ARI — Alignment between Embedding Clusters and Level Labels / Embedding 군집과 Level 라벨 일치도

> **PRD §3.2.3**: ARI ≥ 0.6 indicates refinement completion criteria is met.  
> ARI ≥ 0.6 이면 정제 완료 기준 충족

In [ ]:
def compute_ari(embeddings: np.ndarray,
                true_labels: np.ndarray,
                n_clusters: int = 6,
                random_state: int = 42) -> float:
    """
    Embedding에 k-means 클러스터링 적용 후 Level 라벨과의 ARI 산출.
    Applies k-means clustering to embeddings and computes ARI against Level labels.

    PRD §3.2.3:
        ARI ≥ 0.6 → Phase 1 정제 완료 기준 충족 / Phase 1 refinement completion criteria met
    """
    km             = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    cluster_labels = km.fit_predict(embeddings)
    ari            = adjusted_rand_score(true_labels, cluster_labels)
    return float(ari)


def compute_silhouette(embeddings: np.ndarray, labels: np.ndarray) -> float:
    """
    Silhouette Score 산출 / Compute Silhouette Score.

    사용 목적 / Purpose:
    - labels = Level (0~5): Level 군집의 분리도 / Level cluster separation
    - labels = color (Y/M/C/K): 색상별 분리도 → PRD §5.3.1 독립 모델 판단 / Per-color separation → independent model decision
    """
    if len(set(labels)) < 2:
        return 0.0
    # 샘플 수가 많으면 subsample (속도) / Subsample for speed when sample count is large
    n = len(embeddings)
    if n > 5000:
        idx        = np.random.choice(n, 5000, replace=False)
        embeddings = embeddings[idx]
        labels     = labels[idx]
    return float(silhouette_score(embeddings, labels, sample_size=min(n, 3000), random_state=42))


print("=" * 50)
print("정량 지표 계산 중 / Computing quantitative metrics...")

level_labels     = df_viz["level"].values
color_labels_enc = LabelEncoder().fit_transform(df_viz["color"].values)

# ── Level 기준 지표 / Level-based metrics ──────────────────────────────
ari_all   = compute_ari(embeddings, level_labels, n_clusters=6)
sil_level = compute_silhouette(embeddings, level_labels)

# ── 색상별 독립 지표 (PRD §5.3.1) / Per-color independence metrics ──────
sil_color = compute_silhouette(embeddings, color_labels_enc)

print(f"\n{'='*50}")
print(f"[전체 / Global] ARI (Level vs k-means)  : {ari_all:.4f}")
print(f"  → 목표 / Target: ≥ {ARI_THRESHOLD} "
      f"| {'달성 / Met' if ari_all >= ARI_THRESHOLD else '미달 / Not met — Phase 1 추가 정제 필요 / Additional refinement needed'}")
print(f"[전체 / Global] Silhouette (Level)      : {sil_level:.4f}")
print(f"[전체 / Global] Silhouette (CMYK 색상 / Color)  : {sil_color:.4f}")
print(f"  → 독립 모델 분리 기준 / Independent model threshold: ≥ {SILHOUETTE_THRESHOLD} "
      f"| {'색상별 독립 모델 검토 권장 / Per-color independent model recommended' if sil_color >= SILHOUETTE_THRESHOLD else '공유 모델 적합 / Shared model suitable'}")

# ── 색상별 ARI / Per-color ARI ─────────────────────────────────────────
ari_by_color = {}
sil_by_color = {}
print("\n[색상별 지표 / Per-color metrics]")
for color_code in ["Y", "M", "C", "K"]:
    mask  = df_viz["color"].values == color_code
    emb_c = embeddings[mask]
    lv_c  = level_labels[mask]
    if len(emb_c) < 10:
        continue
    ari_c                  = compute_ari(emb_c, lv_c)
    sil_c                  = compute_silhouette(emb_c, lv_c)
    ari_by_color[color_code] = ari_c
    sil_by_color[color_code] = sil_c
    print(f"  {color_code}: ARI={ari_c:.4f}  Silhouette(Level)={sil_c:.4f}")

### 5.2 Label-Embedding Mismatch Analysis / 라벨-Embedding 불일치 샘플 분석

> **PRD §5.6.4**: Extract samples where human labels (Level) and embedding distance are misaligned.  
> 인간 라벨(Level)과 embedding 거리 간 불일치 샘플 추출  
> **PRD §3.2.3**: Mismatch sample ratio ≤ 10% / 불일치 샘플 비율 ≤ 10%

In [ ]:
def find_mismatch_samples(df: pd.DataFrame,
                           embeddings: np.ndarray,
                           n_clusters: int = 6,
                           random_state: int = 42) -> pd.DataFrame:
    """
    k-means 클러스터 할당과 Level 라벨이 불일치하는 샘플을 추출한다.
    Extracts samples where k-means cluster assignment and Level label are mismatched.

    불일치 기준 / Mismatch criterion:
        각 k-means 클러스터에서 가장 빈번한 Level = 대표 Level로 간주.
        The most frequent Level in each k-means cluster is treated as the representative Level.
        해당 클러스터에서 대표 Level이 아닌 샘플 = 불일치 샘플.
        Samples in that cluster that are not the representative Level = mismatch samples.

    반환 컬럼 / Return columns:
        filename, color, level (실제 / actual), cluster_level (클러스터 대표 / cluster representative),
        confidence, tsne_x, tsne_y
    """
    km          = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    cluster_ids = km.fit_predict(embeddings)

    df              = df.copy()
    df["cluster_id"] = cluster_ids

    # 클러스터별 대표 Level 결정 / Determine representative Level per cluster
    cluster_mode = (
        df.groupby("cluster_id")["level"]
        .agg(lambda x: x.mode().iloc[0])
        .rename("cluster_level")
    )
    df = df.join(cluster_mode, on="cluster_id")

    mismatch              = df[df["level"] != df["cluster_level"]].copy()
    mismatch["level_diff"] = (mismatch["level"] - mismatch["cluster_level"]).abs()
    mismatch              = mismatch.sort_values("level_diff", ascending=False)

    return mismatch


df_mismatch = find_mismatch_samples(df_viz, embeddings)

mismatch_ratio = len(df_mismatch) / len(df_viz)
print(f"\n불일치 샘플 수 / Mismatch count: {len(df_mismatch)} / {len(df_viz)} "
      f"({mismatch_ratio * 100:.1f}%)")
print(f"  → 기준 / Threshold: ≤ {MISMATCH_THRESHOLD * 100:.0f}% "
      f"| {'기준 충족 / Met' if mismatch_ratio <= MISMATCH_THRESHOLD else '기준 초과 / Exceeded — 라벨 정제 필요 / Label refinement needed'}")

# 색상별 불일치 현황 / Per-color mismatch status
print("\n[색상별 불일치 현황 / Per-color mismatch status]")
print(df_mismatch.groupby("color").size().to_string())

# 저장 / Save
mismatch_path = OUTPUT_DIR / "mismatch_samples.csv"
df_mismatch.to_csv(mismatch_path, index=False, encoding="utf-8-sig")
print(f"\n[저장 / Saved] {mismatch_path}")

df_mismatch.head(10)

### 5.3 Mismatch Sample Visualization / 불일치 샘플 시각화

> Highlight mismatch samples with a star marker (★) in t-SNE space.  
> t-SNE 공간에서 불일치 샘플을 별도 마커(★)로 강조 표시

In [ ]:
def plot_mismatch_overlay(df_all: pd.DataFrame,
                           df_mismatch: pd.DataFrame,
                           x_col: str, y_col: str,
                           level_colors: List[str],
                           output_path: Optional[str] = None) -> go.Figure:
    """
    불일치 샘플을 ★ 마커로 overlay한 t-SNE scatter plot.
    t-SNE scatter plot with mismatch samples overlaid as ★ markers.
    """
    color_map = {str(i): c for i, c in enumerate(level_colors)}

    fig = px.scatter(
        df_all, x=x_col, y=y_col,
        color="level_str",
        color_discrete_map=color_map,
        title="t-SNE — Mismatch Samples Highlighted (★) / 불일치 샘플 강조",
        template="plotly_dark",
        hover_data=["filename", "color", "level"],
        width=900, height=650,
    )
    fig.update_traces(marker=dict(size=6, opacity=0.5))

    # 불일치 샘플 overlay / Overlay mismatch samples
    if len(df_mismatch) > 0 and x_col in df_mismatch.columns:
        fig.add_trace(go.Scatter(
            x=df_mismatch[x_col],
            y=df_mismatch[y_col],
            mode="markers",
            name="Mismatch (★) / 불일치 샘플",
            marker=dict(
                symbol="star",
                size=14,
                color="#ff7aa2",
                line=dict(width=1, color="white"),
                opacity=0.95,
            ),
            text=df_mismatch.apply(
                lambda r: f"{r['filename']}<br>실제 / Actual:{r['level']} 클러스터 / Cluster:{r['cluster_level']}",
                axis=1
            ),
            hovertemplate="%{text}<extra></extra>",
        ))

    fig.update_layout(
        font=dict(family="Segoe UI", size=12),
        margin=dict(l=40, r=40, t=60, b=40),
    )

    if output_path:
        fig.write_html(output_path)
        print(f"[저장 / Saved] {output_path}")
    return fig


df_mismatch_with_coords = df_mismatch.copy()

fig_mismatch = plot_mismatch_overlay(
    df_viz, df_mismatch_with_coords,
    "tsne_x", "tsne_y",
    level_colors=LEVEL_COLORS,
    output_path=str(OUTPUT_DIR / "tsne_mismatch_overlay.html"),
)
fig_mismatch.show()

---
## 6. Metrics Summary & Phase 1 Decision / 지표 요약 및 Phase 1 판단

> **PRD §3.2.3**: Automatic evaluation of refinement completion criteria.  
> 정제 완료 판단 기준 자동 평가

In [ ]:
# ── 지표 집계 / Metrics aggregation ───────────────────────────────────
metrics_summary = {
    "meta": {
        "n_samples":  int(len(df_viz)),
        "backbone":   BACKBONE_NAME,
        "checkpoint": str(CHECKPOINT) if CHECKPOINT else None,
        "tsne_kl":    None,
    },
    "global": {
        "ARI":                           round(ari_all, 4),
        "ARI_threshold":                 ARI_THRESHOLD,
        "ARI_pass":                      ari_all >= ARI_THRESHOLD,
        "silhouette_level":              round(sil_level, 4),
        "silhouette_color":              round(sil_color, 4),
        "silhouette_color_threshold":    SILHOUETTE_THRESHOLD,
        "independent_model_recommended": sil_color >= SILHOUETTE_THRESHOLD,
        "mismatch_count":                int(len(df_mismatch)),
        "mismatch_ratio":                round(mismatch_ratio, 4),
        "mismatch_threshold":            MISMATCH_THRESHOLD,
        "mismatch_pass":                 mismatch_ratio <= MISMATCH_THRESHOLD,
    },
    "by_color": {
        color: {
            "ARI":        round(ari_by_color.get(color, -1), 4),
            "silhouette": round(sil_by_color.get(color, -1), 4),
        }
        for color in ["Y", "M", "C", "K"]
    },
}

# 저장 / Save
metrics_path = OUTPUT_DIR / "metrics_summary.json"
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics_summary, f, ensure_ascii=False, indent=2)
print(f"[저장 / Saved] {metrics_path}")

# ── Phase 1 판단 출력 / Phase 1 decision output ────────────────────────
print("\n" + "=" * 60)
print("Phase 1 Refinement Completion Criteria / 정제 완료 판단 (PRD §3.2.3)")
print("=" * 60)

cond_ari      = metrics_summary["global"]["ARI_pass"]
cond_mismatch = metrics_summary["global"]["mismatch_pass"]

print(f"  [1] ARI ≥ {ARI_THRESHOLD}         : "
      f"{ari_all:.4f}  →  {'충족 / Met' if cond_ari else '미달 / Not met'}")
print(f"  [2] 불일치 비율 / Mismatch ≤ {MISMATCH_THRESHOLD*100:.0f}%  : "
      f"{mismatch_ratio*100:.1f}%  →  {'충족 / Met' if cond_mismatch else '미달 / Not met'}")
print()

if cond_ari and cond_mismatch:
    print("  [PASS] Phase 1 정제 완료 기준 충족 → Phase 2 진입 가능")
    print("         Phase 1 refinement criteria met → Ready to enter Phase 2")
else:
    print("  [FAIL] 기준 미달 → 불일치 샘플 검토 후 라벨 재정제 필요")
    print("         Criteria not met → Review mismatch samples and refine labels")
    print("         mismatch_samples.csv 확인 후 Phase 1 재수행 / Re-run Phase 1 after reviewing")

print()
independent = metrics_summary["global"]["independent_model_recommended"]
print(f"  [3] 색상별 독립 모델 검토 / Per-color independent model review (PRD §5.3.1)")
print(f"      Silhouette(CMYK) = {sil_color:.4f}  "
      f"{'≥' if independent else '<'} {SILHOUETTE_THRESHOLD}")
print(f"      → {'색상별 독립 모델 분리 검토 권장 / Per-color independent model recommended' if independent else '공유 모델 적합 / Shared model suitable'}")

---
## 7. Metrics Dashboard / 지표 시각화 대시보드

In [ ]:
def plot_metrics_dashboard(metrics: Dict[str, dict],
                            output_path: Optional[str] = None) -> go.Figure:
    """
    ARI, Silhouette, 불일치 비율을 Gauge + Bar로 시각화.
    Visualizes ARI, Silhouette, and mismatch ratio as Gauge + Bar charts.
    """
    g            = metrics["global"]
    colors_order = ["Y", "M", "C", "K"]

    fig = make_subplots(
        rows=2, cols=3,
        specs=[
            [{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}],
            [{"type": "bar",       "colspan": 3},     None,                    None],
        ],
        subplot_titles=[
            "ARI (Level vs k-means)",
            "Mismatch Ratio / 불일치 샘플 비율",
            "Silhouette (CMYK Color / 색상)",
            "Per-color ARI / 색상별 ARI",
        ],
        vertical_spacing=0.15,
    )

    # Gauge: ARI
    fig.add_trace(go.Indicator(
        mode="gauge+number",
        value=g["ARI"],
        number={"font": {"size": 32}},
        gauge={
            "axis": {"range": [0, 1]},
            "bar": {"color": "#50e3c2"},
            "threshold": {"line": {"color": "#ff7aa2", "width": 3},
                           "value": g["ARI_threshold"]},
            "bgcolor": "#0b1220",
        },
        title={"text": f"Target / 목표 ≥ {g['ARI_threshold']}"},
    ), row=1, col=1)

    # Gauge: 불일치 비율 (낮을수록 좋음) / Mismatch ratio (lower is better)
    fig.add_trace(go.Indicator(
        mode="gauge+number",
        value=round(g["mismatch_ratio"] * 100, 1),
        number={"suffix": "%", "font": {"size": 32}},
        gauge={
            "axis": {"range": [0, 50]},
            "bar": {"color": "#66d9ff"},
            "threshold": {"line": {"color": "#ff7aa2", "width": 3},
                           "value": g["mismatch_threshold"] * 100},
            "bgcolor": "#0b1220",
        },
        title={"text": f"Target / 목표 ≤ {g['mismatch_threshold']*100:.0f}%"},
    ), row=1, col=2)

    # Gauge: Silhouette (색상 / color)
    fig.add_trace(go.Indicator(
        mode="gauge+number",
        value=g["silhouette_color"],
        number={"font": {"size": 32}},
        gauge={
            "axis": {"range": [-1, 1]},
            "bar": {"color": "#c792ea"},
            "threshold": {"line": {"color": "#ffb347", "width": 3},
                           "value": g["silhouette_color_threshold"]},
            "bgcolor": "#0b1220",
        },
        title={"text": f"Independent model threshold / 독립 모델 기준 ≥ {g['silhouette_color_threshold']}"},
    ), row=1, col=3)

    # Bar: 색상별 ARI / Per-color ARI
    by_color   = metrics["by_color"]
    bar_colors = ["#f5e642", "#e91e8c", "#00b4d8", "#aaaaaa"]
    fig.add_trace(go.Bar(
        x=colors_order,
        y=[by_color[c]["ARI"] for c in colors_order],
        marker_color=bar_colors,
        text=[f"{by_color[c]['ARI']:.3f}" for c in colors_order],
        textposition="outside",
        name="ARI",
    ), row=2, col=1)

    # 목표선 / Target line
    fig.add_hline(y=g["ARI_threshold"], line_dash="dot",
                   line_color="#ff7aa2",
                   annotation_text=f"Target / 목표 {g['ARI_threshold']}",
                   row=2, col=1)

    fig.update_layout(
        title="Embedding Quality Metrics Dashboard / Embedding 품질 지표 대시보드",
        title_font=dict(size=18),
        template="plotly_dark",
        font=dict(family="Segoe UI", size=12),
        width=1100, height=700,
        margin=dict(l=40, r=40, t=80, b=40),
        showlegend=False,
    )

    if output_path:
        fig.write_html(output_path)
        print(f"[저장 / Saved] {output_path}")
    return fig


fig_dashboard = plot_metrics_dashboard(
    metrics_summary,
    output_path=str(OUTPUT_DIR / "metrics_dashboard.html"),
)
fig_dashboard.show()

---
## 8. Output Summary / 산출물 요약

In [ ]:
print("=" * 60)
print("Generated Outputs / 생성된 산출물 목록")
print("=" * 60)

output_files = [
    ("tsne_by_level.html",          "t-SNE — Level-colored (interactive) / Level별 색상 코딩"),
    ("tsne_by_color.html",          "t-SNE — Per-CMYK distribution (interactive) / CMYK 채널별 분포"),
    ("tsne_per_color_subplot.html", "t-SNE — CMYK 4-panel subplot / CMYK 4분할 서브플롯"),
    ("tsne_mismatch_overlay.html",  "t-SNE — Mismatch samples highlighted (★) / 불일치 샘플 강조"),
    ("umap_by_level.html",          "UMAP — Level-colored (requires umap-learn) / Level별 색상 코딩"),
    ("metrics_dashboard.html",      "Embedding quality metrics dashboard / 지표 대시보드"),
    ("metrics_summary.json",        "ARI / Silhouette / Mismatch ratio / 불일치 비율 수치"),
    ("mismatch_samples.csv",        "Label-Embedding mismatch sample list / 불일치 샘플 목록"),
]

for fname, desc in output_files:
    full_path = OUTPUT_DIR / fname
    exists    = "[OK]" if full_path.exists() else "[  ]"
    print(f"  {exists}  {fname:<40} {desc}")

print()
print("Next steps / 다음 단계 (PRD §3.2.2):")
print("  1. Open tsne_by_level.html and visually check Level cluster boundaries")
print("     tsne_by_level.html 을 열고 Level 군집 경계를 육안으로 확인")
print("  2. Review mismatch_samples.csv in order and correct labels")
print("     mismatch_samples.csv 의 불일치 샘플을 순서대로 검토하여 라벨 수정")
print("  3. Check ARI / mismatch ratio criteria in metrics_dashboard")
print("     metrics_dashboard 에서 ARI / 불일치 비율 기준 충족 여부 확인")
print("  4. If criteria not met → correct labels and re-run this notebook (Phase 1 loop)")
print("     기준 미달 시 → 라벨 수정 후 이 노트북 재실행 (Phase 1 루프)")
print("  5. If criteria met → pass refined CSV to notebooks/03_training.ipynb")
print("     기준 충족 시 → 정제된 CSV를 notebooks/03_training.ipynb 에 전달")